# 29장 · 위상 데이터 분석(TDA) — 점 구름의 모양을 바코드로

이산위상 연구실 활용 편 · 글 안상선. 그대로 위에서부터 실행하면 돼요.

In [1]:
# 처음 한 번만: 필요한 패키지 설치 (구글 코랩이면 앞에 !를 붙여 그대로 실행)
# pip install numpy scipy matplotlib scikit-image networkx trimesh ripser

잡음 섞인 원 위의 점 구름과 아무 모양 없는 점 구름에 **지속 호몰로지**(ripser)를 계산해, 오래 버티는 구멍 막대(H₁)가 있는지 봅니다.

In [2]:
import numpy as np
from ripser import ripser

rng = np.random.default_rng(5)
n = 60
t = 2 * np.pi * rng.random(n)
circle = np.column_stack([np.cos(t), np.sin(t)]) + 0.12 * rng.standard_normal((n, 2))
blob = 0.6 * rng.standard_normal((n, 2))

for name, X in [('원 + 잡음', circle), ('뭉친 점', blob)]:
    h1 = ripser(X)['dgms'][1]
    life = (h1[:, 1] - h1[:, 0]) if len(h1) else np.array([0.0])
    print(f"{name:<6} H1 막대 {len(h1):>2}개, 가장 긴 막대 {life.max():.2f}, 둘째 {np.sort(life)[-2] if len(life) > 1 else 0:.2f}")

원 + 잡음 H1 막대  7개, 가장 긴 막대 0.88, 둘째 0.06
뭉친 점   H1 막대 10개, 가장 긴 막대 0.17, 둘째 0.08


원은 막대 하나가 압도적으로 길고(진짜 구멍), 뭉친 점은 짧은 막대뿐(잡음)입니다. 이번에는 이산 모스 이론의 생각 — **중요한 자리만 남기기** — 을 1차원 땅 높이로 확인합니다.

In [3]:
h = [6,3,5,8,2,4,7,9,5,1,3,6,8,4,6,10,7,3,5,8]     # 페이지의 「물 빠지기」 실험과 같은 땅

def sublevel_pairs(h):
    """물 높이를 올리며 웅덩이가 생기고(골짜기) 합쳐지는(봉우리) 높이를 짝짓기 — 나이 많은 웅덩이가 살아남음"""
    n = len(h); parent = list(range(n)); born = {}; active = set(); pairs = []
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in sorted(range(n), key=lambda i: (h[i], i)):
        active.add(i); born[i] = h[i]
        for j in (i - 1, i + 1):
            if j in active:
                a, b = find(i), find(j)
                if a == b: continue
                young, old = (a, b) if born[a] > born[b] else (b, a)
                if born[young] < h[i]: pairs.append((born[young], h[i]))
                parent[young] = old
    pairs.append((min(h), float('inf')))
    return sorted(pairs)

P = sublevel_pairs(h)
mins = sum(1 for i, v in enumerate(h) if (i == 0 or h[i-1] > v) and (i == len(h)-1 or h[i+1] > v))
print('웅덩이 막대 (생김 → 합쳐짐):', P)
print(f'땅 {len(h)}자리 중 골짜기 바닥 {mins}곳만 막대를 만든다 → 나머지는 계산에서 빼도 됨 (이산 모스의 생각)')

웅덩이 막대 (생김 → 합쳐짐): [(1, inf), (2, 9), (3, 8), (3, 10), (4, 8)]
땅 20자리 중 골짜기 바닥 5곳만 막대를 만든다 → 나머지는 계산에서 빼도 됨 (이산 모스의 생각)
